In [70]:
import netCDF4 as nc
import numpy as np
from pathlib import Path

def open_dataset(path_str):
    """Open a NetCDF dataset from the given path."""
    path = Path(path_str).expanduser().resolve()
    if not path.is_file():
        raise FileNotFoundError(f"File not found: {path}")
    return nc.Dataset(path, mode="r")

def normalize_time_index(time_index, ntime):
    """
    Normalize a user-provided time index.
    - Converts to int
    - Allows -1 to mean the last time index
    - Checks bounds
    """
    idx = int(time_index)
    if idx == -1:
        idx = ntime - 1

    if idx < 0 or idx >= ntime:
        raise IndexError(
            f"time index {idx} out of bounds [0, {ntime - 1}] "
            "(note: -1 means the last index before normalization)"
        )
    return idx

def compute_l1_l2_error(state_target, state_ref):
    """
    Compute L1, L2, and Lmax errors of state_target relative to state_ref.

    Returns: (l1, l2, lmax)
    """
    diff = state_target - state_ref
    diff_flat = diff.ravel().astype(np.float64)

    l1 = np.mean(np.abs(diff_flat))
    l2 = np.sqrt(np.mean(diff_flat**2))
    lmax = np.max(np.abs(diff_flat))

    return l1, l2, lmax

In [71]:
# === USER INPUT SECTION ===

# 1. Define File Paths
# (You can use suffixes like in your previous notebook, or full paths)
base_path = "/home/amir/Projects/Python/Atmpy/atmpy/output_data/baroclinic_qg_vortex/_baroclinic_qg_vortex_40_20_40"
file_1_suffix = "_N_50_cor_500_QG_off_p_5_Hfac_4_rossby.nc"
file_2_suffix = "_N_50_cor_500_QG_on_p_5_Hfac_4_rossby.nc"

file1_path = base_path + file_1_suffix
file2_path = base_path + file_2_suffix

# 2. Variable to analyze
var_name = "rho"

# 3. Time Indices
# Specify which time step to grab from each file.
# 0  = First time step
# -1 = Last time step
# Any other integer = specific index

time_index_1 = -1   # <--- Time index for File 1
time_index_2 = -1    # <--- Time index for File 2

print("File 1 Path:", file1_path)
print("File 2 Path:", file2_path)
print("Variable:", var_name)
print(f"Comparing File 1 (Time Index: {time_index_1}) vs File 2 (Time Index: {time_index_2})")

File 1 Path: /home/amir/Projects/Python/Atmpy/atmpy/output_data/baroclinic_qg_vortex/_baroclinic_qg_vortex_40_20_40_N_50_cor_500_QG_off_p_5_Hfac_4_rossby.nc
File 2 Path: /home/amir/Projects/Python/Atmpy/atmpy/output_data/baroclinic_qg_vortex/_baroclinic_qg_vortex_40_20_40_N_50_cor_500_QG_on_p_5_Hfac_4_rossby.nc
Variable: rho
Comparing File 1 (Time Index: -1) vs File 2 (Time Index: -1)


In [72]:
# Open datasets
ds1 = open_dataset(file1_path)
ds2 = open_dataset(file2_path)
print("Datasets opened successfully.\n")

def get_data_at_time(ds, label, var_name, t_idx_input):
    """Helper to extract data at a specific time index"""
    if var_name not in ds.variables:
        raise KeyError(f"Variable '{var_name}' not found in {label}")

    var = ds.variables[var_name]
    ntime = var.shape[0]

    # Normalize index (handle -1)
    idx = normalize_time_index(t_idx_input, ntime)

    # Extract data (Time is assumed to be the first dimension)
    data = np.array(var[idx, ...])
    return data, idx, var[0,...].shape

try:
    # 1. Get State from File 1
    state1, idx1, shape1 = get_data_at_time(ds1, "FILE 1", var_name, time_index_1)

    # 2. Get State from File 2
    state2, idx2, shape2 = get_data_at_time(ds2, "FILE 2", var_name, time_index_2)

    print("-" * 50)
    print(f"=== CROSS-FILE COMPARISON ===")
    print(f"File 1 (Time: {idx1})  vs  File 2 (Time: {idx2})")
    print(f"Variable: {var_name}")
    print(f"Shape 1:  {shape1}")
    print(f"Shape 2:  {shape2}")

    # 3. Compute Errors
    if shape1 != shape2:
        print("\nERROR: Shapes do not match! Cannot compute direct difference.")
    else:
        l1, l2, lmax = compute_l1_l2_error(state1, state2)

        print("\nResults (State1 - State2):")
        print(f"L1 Error (Mean Absolute Diff): {l1:e}")
        print(f"L2 Error (Root Mean Square):   {l2:e}")
        print(f"Lmax Error (Max Absolute):     {lmax:e}")

        if lmax == 0.0:
            print("\n>>> Conclusion: The states are IDENTICAL.")
        else:
            print("\n>>> Conclusion: The states are DIFFERENT.")

finally:
    # Always close files to prevent locking
    ds1.close()
    ds2.close()
    print("\nDatasets closed.")

Datasets opened successfully.

--------------------------------------------------
=== CROSS-FILE COMPARISON ===
File 1 (Time: 2448)  vs  File 2 (Time: 495)
Variable: rho
Shape 1:  (40, 20, 40)
Shape 2:  (40, 20, 40)

Results (State1 - State2):
L1 Error (Mean Absolute Diff): 6.429928e-07
L2 Error (Root Mean Square):   1.249902e-06
Lmax Error (Max Absolute):     1.325780e-05

>>> Conclusion: The states are DIFFERENT.

Datasets closed.
